# Entrenamiento unificado CubiCasa5K + FloorPlanCAD (YOLOv8-seg)

Este notebook reemplaza a `COLAB_ENTRENAMIENTO_YOLO_SEG.ipynb` (el que generó
`best_seg.pt`, con mAP50=0.09 -- su celda de labels escribía solo el número
de clase sin coordenadas, un bug real, no un problema de hiperparámetros).

Entrena un solo modelo YOLOv8-seg con un vocabulario de clases unificado:
- **CubiCasa5K** (dataset real, 5000 planos, vía `training/cubicasa/`): wall, room,
  bathroom, kitchen, living_room, bedroom, dining_room, outdoor, door, window,
  column, stairs, toilet, sink, bathtub, shower, wardrobe, cabinet,
  kitchen_counter, fireplace, electrical_appliance, bench, misc_furniture.
- **FloorPlanCAD** (200 planos, ya en el repo vía `best_floorplancad.pt`):
  aporta `bed, sofa, table, chair, refrigerator, gas_stove` -- CubiCasa5K NO
  tiene mobiliario movible anotado (verificado contra el dataset real).

**Tiempo esperado (T4 gratis):**
- Descarga de CubiCasa5K real: ~5.5 GB, puede tardar 20-60 min según la hora.
- Construcción del dataset (parseo de ~5000 SVGs): ~10-20 min.
- Descarga + conversión de FloorPlanCAD: ~10-15 min.
- Entrenamiento (yolov8s-seg, ~50-80 épocas): varias horas -- Colab gratis
  desconecta sesiones inactivas, revisa la pestaña de vez en cuando.

**Antes de correr:** subí una copia del repo (o al menos la carpeta
`training/` completa + `app/` si querés probar el modelo después) a Google
Drive, y ajustá `PROJECT_DIR` en la celda 2.

In [ ]:
import torch
print("Torch version:", torch.__version__)
if not torch.cuda.is_available():
    print("ADVERTENCIA: no se detecto GPU. Entorno de ejecucion > Cambiar tipo de entorno > T4 GPU.")
else:
    print(f"GPU activa: {torch.cuda.get_device_name(0)}")


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Ajusta esto a donde subiste el repo dentro de tu Drive.
PROJECT_DIR = "/content/drive/MyDrive/Backend-Python"

import os
os.chdir(PROJECT_DIR)
print("cwd:", os.getcwd())


## 1. Instalar dependencias

In [ ]:
!pip install -q ultralytics lxml requests tqdm huggingface_hub


## 2. Descargar y construir el dataset CubiCasa5K real

Descarga el zip crudo (Zenodo, ~5.5 GB) y corre el pipeline de
`training/cubicasa/` (parser de SVG + mapeo de clases + emisor de labels
seg) sobre las ~5000 muestras. Esto es exactamente lo que se validó
localmente contra ~60 muestras reales antes de escribir este notebook.

In [ ]:
!python -m training.download_cubicasa_real --output-dir YOLO/cubicasa5k_raw


In [ ]:
!python -m training.cli build \
  --cubicasa-root YOLO/cubicasa5k_raw/cubicasa5k \
  --train-split YOLO/cubicasa5k_raw/cubicasa5k/train.txt \
  --val-split YOLO/cubicasa5k_raw/cubicasa5k/val.txt \
  --test-split YOLO/cubicasa5k_raw/cubicasa5k/test.txt \
  --output-root YOLO/cubicasa5k_seg


## 3. Descargar y convertir FloorPlanCAD

Aporta las clases de mobiliario movible (`bed, sofa, table, chair, ...`)
que CubiCasa5K no tiene. Si `YOLO/floorplancad/` ya existe en tu copia del
repo (subida a Drive), podés saltarte esta celda.

In [ ]:
!python -m training.download_floorplancad_hf --all
!python -m training.convert_floorplancad_to_yolo


## 4. Fusionar ambos datasets en un vocabulario unificado

In [ ]:
!python -m training.build_cubicasa_floorplancad_mix \
  --cubicasa YOLO/cubicasa5k_seg \
  --floorplancad YOLO/floorplancad \
  --output YOLO/unified_train


In [ ]:
import yaml
with open("YOLO/unified_train/data.yaml") as f:
    data_yaml = yaml.safe_load(f)
print(f"{len(data_yaml['names'])} clases:", data_yaml["names"])


## 5. Entrenar YOLOv8-seg

Mismos hiperparámetros base que ya funcionaron en
`training/entrenamiento_cubicasa_colab.ipynb` (el notebook que sí produjo un
modelo bueno, mAP50=0.69), ajustables. `yolov8s-seg` en vez de `yolov8n-seg`
porque ahora hay más clases (29) que distinguir.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8s-seg.pt")
results = model.train(
    data="YOLO/unified_train/data.yaml",
    epochs=80,
    imgsz=832,
    batch=16,
    patience=15,
    device=0,
    project="/content/runs",
    name="unified_cubicasa_floorplancad",
    exist_ok=True,
    save=True,
    verbose=True,
    lr0=0.01,
    lrf=0.01,
    warmup_epochs=3,
)


## 6. Guardar el modelo resultante a Drive

Instalar en el backend: copiar el `.pt` a `YOLO/best_unified_seg.pt`,
apuntar `YOLO_MODEL_PATH` en `app/core/config.py` (o la variable de entorno)
a esa ruta, y **revisar `app/services/furniture_detector.py`**: su
`CLASS_TO_UNITY_TYPE` tiene que cubrir las 29 clases nuevas de
`YOLO/unified_train/data.yaml`, no solo las 15 de antes -- esa reconexión
queda pendiente para la siguiente sesión, una vez haya pesos reales para
probar contra el backend.

In [ ]:
import shutil
from pathlib import Path

model_source = Path("/content/runs/unified_cubicasa_floorplancad/weights/best.pt")
drive_dest = Path("/content/drive/MyDrive/best_unified_seg.pt")

if model_source.exists():
    shutil.copy(model_source, drive_dest)
    print(f"Guardado en Drive: {drive_dest} ({drive_dest.stat().st_size / 1e6:.1f} MB)")
else:
    print("No se encontro el modelo entrenado -- revisa si el entrenamiento termino bien.")
